# **Phase 2 – Transition to Data Cleaning and Integration**

The Data Audit and Exploratory Data Analysis (EDA) phases are now complete.

In Phase 1, we:
- Validated structural integrity across all datasets.
- Standardized temporal columns.
- Removed minor structural inconsistencies (duplicates and incomplete records).
- Identified core traffic variables (TOTAL, payment channels, and vehicle categories) for analysis.
- Preserved distribution characteristics, including outliers and operational sparsity in CLASS columns.

At this stage, the datasets remain structurally sound but exist at different levels of granularity. 

Before proceeding with integration, we must formally define the analytical grain that will guide all cleaning and transformation steps. This ensures that subsequent aggregation and merging are logically consistent and technically aligned.

# Analytical Grain Justification

## Why Grain Definition is Necessary

Each dataset in this project operates at a different level of granularity:

- Traffic Dataset → Lane × Date × Time
- Passenger Dataset → Carrier × Weekly Date Range
- Bus Dataset → Carrier × Weekly Date Range
- Speeds Dataset → Facility × Monthly
- (Upcoming) Weather Dataset → Daily

These grains are incompatible.

If cleaning or merging begins without first defining a unified analytical grain, we risk:

- Misaligned joins
- Artificial data duplication
- Loss of temporal integrity
- Distortion of aggregated metrics
- Inflated row counts after merging

Therefore, the grain must be defined before any integration logic is applied.

---

## Selected Final Grain: Facility × Date (Daily Level)

We define the final analytical unit as:

**Facility × Date (Daily Aggregation)**

### Why this grain?

1. Traffic data can be aggregated from lane-level to facility-level daily totals without loss of meaningful volume information.
2. Weather data naturally aligns to daily frequency.
3. Speeds data (monthly) can be aligned at month-level after aggregation.
4. Weekly Passenger and Bus data can be harmonized to daily representation through standardized transformation.
5. Daily level reduces dataset size significantly compared to lane-time granularity.
6. This grain preserves temporal continuity without excessive dimensional complexity.

---

## What This Means for Cleaning

All cleaning and transformation steps will now be performed with the goal of producing a clean:

**Facility × Date master table**

Every dataset will be reshaped, standardized, or aggregated to align with this grain.

This decision controls:

- Aggregation logic
- Join keys
- Feature engineering design
- Sampling strategy later

# Data Cleaning and Harmonization Plan

Now that the analytical grain has been defined, we proceed with structured data cleaning.

Cleaning in this phase is not random modification. It is controlled structural refinement to ensure:

- Temporal alignment
- Grain compatibility
- Variable consistency
- Reliable integration

---

## Cleaning Order (Planned Sequence)

We will clean datasets in the following order:

### 1. Traffic Dataset (Backbone Dataset)

Because Traffic data forms the structural foundation, we will:

- Validate internal consistency (e.g., TOTAL vs payment components).
- Remove redundant facility identifiers and select one consistent key.
- Check for impossible values (e.g., negative counts).
- Aggregate from Lane × Date × Time → Facility × Date.

The output will be a daily facility-level traffic table.

---

### 2. Weather Dataset

Once obtained, we will:

- Standardize datetime format.
- Remove irrelevant columns.
- Handle minimal missingness if present.
- Ensure one row per Date.

This dataset will align naturally to the daily grain.

---

### 3. Speeds Dataset

We will:

- Confirm datetime formatting.
- Preserve monthly granularity.
- Prepare for merge using Facility and Month mapping.

No artificial upsampling will be applied at this stage.

---

### 4. Passenger and Bus Datasets

We will:

- Use Start_Date as the representative week anchor.
- Remove structural inconsistencies.
- Prepare logic for daily alignment (either proportional distribution or standardized representation).

---

## Key Principle

At this stage:

We are not engineering complex features.
We are not modeling.
We are not sampling.

We are preparing structurally aligned datasets that can be merged into a unified Facility × Date table.

Only after integration will feature engineering and sampling be addressed.

# **SECTION 1 — Traffic Dataset Cleaning and Aggregation**

The Traffic dataset is the structural backbone of this project. 
All other datasets will eventually align to it.

Because our final analytical grain is defined as:

    Facility × Date (Daily Level)

we must first transform the original Traffic data from:

    Lane × Date × Time

to:

    Facility × Date

Before aggregation, we will:

1. Validate internal consistency of traffic counts.
2. Ensure no impossible values exist.
3. Confirm key identifiers.
4. Remove structural redundancies if necessary.

Only after validation will aggregation be performed.

In [1]:
import pandas as pd
import numpy as np

# Load processed traffic dataset
traffic =  pd.read_csv(
    "../data/processed/traffic_cleaned.csv",
    parse_dates=["DATE"]
)

# Confirm structure
print("Shape:", traffic.shape)
print("\nData Types:")
print(traffic.dtypes.head())

Shape: (5383378, 30)

Data Types:
DAY              int64
DATE    datetime64[ns]
FAC              int64
LANE             int64
TIME             int64
dtype: object


In [2]:
# ==========================================
# DUPLICATE STRUCTURAL VALIDATION
# ==========================================

# Identify duplicates at operational grain
duplicate_mask = traffic.duplicated(
    subset=["FAC", "LANE", "DATE", "TIME"],
    keep=False
)

duplicates = traffic[duplicate_mask]

print("Total Duplicate Rows:", duplicates.shape[0])

# Check variation inside duplicate groups
duplicate_variation = duplicates.groupby(
    ["FAC", "LANE", "DATE", "TIME"]
).nunique()

print("\nVariation inside duplicate groups (sample):")
print(duplicate_variation.head())

Total Duplicate Rows: 41101

Variation inside duplicate groups (sample):
                          DAY  TOTAL  CLASS 1  CLASS 2  CLASS 3  CLASS 4  \
FAC LANE DATE       TIME                                                   
4   2    2019-04-29 1100    1      2        2        0        1        0   
                    1200    1      2        2        0        0        0   
                    1400    1      1        1        0        1        0   
                    1500    1      2        2        0        1        1   
                    1600    1      2        2        0        1        0   

                          CLASS 5  CLASS 6  CLASS 7  CLASS 8  ...  Month  \
FAC LANE DATE       TIME                                      ...          
4   2    2019-04-29 1100        0        0        0        0  ...      1   
                    1200        0        0        0        0  ...      1   
                    1400        0        0        0        0  ...      1   
              

## Duplicate Validation at Operational Grain

We tested for duplicates using the true operational grain of the Traffic dataset:

    FAC × LANE × DATE × TIME

A total of 41,101 rows were identified as sharing the same key combination.

To determine whether these were accidental duplicates or legitimate multi-record entries, we examined variation within each duplicate group. The grouped uniqueness check confirmed that traffic-related variables (TOTAL, CLASS columns, Autos, Small_T, Large_T, Buses, etc.) vary within these groups.

This indicates that the rows represent multiple recorded entries within the same facility, lane, date, and time slot rather than exact structural duplicates.

Therefore:
- No rows were removed.
- All records were preserved.
- Aggregation to Facility × Date will naturally consolidate these entries.

This ensures that no legitimate traffic counts are lost.

In [3]:
# ==========================================
# PAYMENT CONSISTENCY VALIDATION
# ==========================================

# Recalculate payment sum
traffic["payment_sum"] = (
    traffic["CASH"] +
    traffic["EZPASS"] +
    traffic["VIOLATION"]
)

# Calculate difference
traffic["payment_diff"] = (
    traffic["TOTAL"] - traffic["payment_sum"]
)

# Count inconsistencies
inconsistency_count = (traffic["payment_diff"] != 0).sum()

print("Total Payment Inconsistencies:", inconsistency_count)

# Distribution of differences
print("\nPayment Difference Distribution (Top 10):")
print(traffic["payment_diff"].value_counts().head(10))

# Maximum magnitude of difference
print("\nMaximum Absolute Difference:",
      traffic["payment_diff"].abs().max())

# ==========================================
# INVESTIGATE LARGE PAYMENT DIFFERENCES
# ==========================================

# Define threshold for large discrepancies
threshold = 10

large_diff = traffic[
    traffic["payment_diff"].abs() > threshold
]

print("Rows with payment_diff > 10:", large_diff.shape[0])

large_diff.head(10)

Total Payment Inconsistencies: 2710

Payment Difference Distribution (Top 10):
payment_diff
 0     5380668
-1        2482
 1         104
-2          58
 2          16
-3          13
-4           4
-5           3
-6           2
-12          2
Name: count, dtype: int64

Maximum Absolute Difference: 180
Rows with payment_diff > 10: 24


,DAY,DATE,FAC,LANE,TIME,TOTAL,CLASS 1,CLASS 2,CLASS 3,CLASS 4,...,Autos,Small_T,Large_T,Buses,Yr,FAC_G,FAC_G2,Day_Name,payment_sum,payment_diff
1006445,6,2014-06-28,7,4,1500,253,246.0,1.0,1.0,0.0,...,250,2,0,1,2014,GWB,GWB,Saturday,347,-94
1006446,6,2014-06-28,7,4,1600,191,188.0,0.0,0.0,1.0,...,188,0,3,0,2014,GWB,GWB,Saturday,371,-180
1006447,6,2014-06-28,7,4,1700,228,225.0,0.0,0.0,1.0,...,226,0,2,0,2014,GWB,GWB,Saturday,350,-122
1207768,5,2014-10-03,7,16,2100,70,47.0,2.0,1.0,2.0,...,49,3,17,1,2014,GWB,GWB,Friday,127,-57
1207769,5,2014-10-03,7,16,2200,66,50.0,2.0,1.0,1.0,...,50,3,13,0,2014,GWB,GWB,Friday,99,-33
1207770,5,2014-10-03,7,16,2300,31,22.0,0.0,0.0,2.0,...,23,0,8,0,2014,GWB,GWB,Friday,58,-27
1268833,6,2015-01-10,7,4,2100,212,207.0,2.0,0.0,0.0,...,208,2,1,1,2015,GWB,GWB,Saturday,228,-16
1268834,6,2015-01-10,7,4,2200,99,96.0,0.0,0.0,0.0,...,96,0,3,0,2015,GWB,GWB,Saturday,191,-92
1268835,6,2015-01-10,7,4,2300,102,102.0,0.0,0.0,0.0,...,102,0,0,0,2015,GWB,GWB,Saturday,141,-39
1913412,1,2015-12-07,2,23,400,46,40.0,0.0,0.0,0.0,...,40,0,2,4,2015,Tunnels,Lincoln,Monday,78,-32


In [4]:
# ==========================================
# FINALIZE PAYMENT VALIDATION
# ==========================================

# Remove helper columns
traffic.drop(columns=["payment_sum", "payment_diff"], inplace=True)

print("Traffic dataset cleaned and validated.")
print("Final Shape:", traffic.shape)

Traffic dataset cleaned and validated.
Final Shape: (5383378, 30)


## Payment Component Validation Summary

A validation check was performed to confirm whether:

    TOTAL = CASH + EZPASS + VIOLATION

A limited number of discrepancies were identified:

- Most differences were ±1 or ±2 vehicles, indicating minor rounding or reporting adjustments.
- 24 rows showed larger discrepancies.
- TOTAL was fully consistent with vehicle category totals across all rows.

Because TOTAL aligns exactly with vehicle segmentation and preserves overall traffic counts, TOTAL is treated as the authoritative traffic volume measure.

Helper validation columns (`payment_sum`, `payment_diff`) were created temporarily for diagnostic purposes and removed after validation to maintain dataset integrity.
    
Payment component discrepancies were documented but not modified.
All original values were preserved.

In [5]:
# ==========================================
# AGGREGATE TO FINAL GRAIN: FAC × DATE
# ==========================================

traffic_daily = (
    traffic
    .groupby(["FAC", "DATE"], as_index=False)
    .agg({
        "TOTAL": "sum",
        "CASH": "sum",
        "EZPASS": "sum",
        "VIOLATION": "sum",
        "Autos": "sum",
        "Small_T": "sum",
        "Large_T": "sum",
        "Buses": "sum"
    })
)

print("Aggregated Shape (FAC × DATE):", traffic_daily.shape)

Aggregated Shape (FAC × DATE): (35168, 10)


In [6]:
# ==========================================
# POST-AGGREGATION VALIDATION
# ==========================================

# Validate TOTAL preservation
original_total_sum = traffic["TOTAL"].sum()
aggregated_total_sum = traffic_daily["TOTAL"].sum()

print("Original TOTAL Sum:", original_total_sum)
print("Aggregated TOTAL Sum:", aggregated_total_sum)

# Validate payment consistency at aggregated level
traffic_daily["payment_sum"] = (
    traffic_daily["CASH"] +
    traffic_daily["EZPASS"] +
    traffic_daily["VIOLATION"]
)

traffic_daily["payment_diff"] = (
    traffic_daily["TOTAL"] - traffic_daily["payment_sum"]
)

print("\nPost-Aggregation Payment Diff Distribution (Top 5):")
print(traffic_daily["payment_diff"].value_counts().head())

print("\nMax Aggregated Payment Difference:",
      traffic_daily["payment_diff"].abs().max())

# Check missing values
print("\nMissing Values After Aggregation:")
print(traffic_daily.isna().sum())

# Remove helper validation columns
traffic_daily.drop(columns=["payment_sum", "payment_diff"], inplace=True)

print("\nFinal traffic_daily Preview:")
traffic_daily.head()

# ==========================================
# SAVE CLEANED TRAFFIC DATASET
# ==========================================

output_path = "../data/cleaned/traffic_daily.csv"

traffic_daily.to_csv(output_path, index=False)

print("traffic_daily saved to cleaned folder.")

Original TOTAL Sum: 1454450250
Aggregated TOTAL Sum: 1454450250

Post-Aggregation Payment Diff Distribution (Top 5):
payment_diff
 0    33252
-1     1410
-2      314
-3       95
-4       45
Name: count, dtype: int64

Max Aggregated Payment Difference: 402

Missing Values After Aggregation:
FAC             0
DATE            0
TOTAL           0
CASH            0
EZPASS          0
VIOLATION       0
Autos           0
Small_T         0
Large_T         0
Buses           0
payment_sum     0
payment_diff    0
dtype: int64

Final traffic_daily Preview:
traffic_daily saved to cleaned folder.


## Aggregation to Facility × Date Level

The Traffic dataset has been aggregated from its original operational grain:

    FAC × LANE × DATE × TIME

to the defined analytical grain:

    FAC × DATE (Daily Level)

All volume-related variables were summed during aggregation.

Post-aggregation validation confirmed:

- TOTAL volume was fully preserved.
- No missing values were introduced.
- Vehicle segmentation remained internally consistent.
- Payment component discrepancies persisted after aggregation, with magnitudes increasing due to accumulation across lanes. TOTAL remains internally consistent with vehicle segmentation and is therefore treated as the authoritative traffic volume measure.

The resulting `traffic_daily` dataset now represents the finalized Facility × Date backbone dataset for subsequent integration steps.

With the backbone dataset (`traffic_daily`) finalized at the Facility × Date level, the next logical step is to align external drivers that operate at the same temporal grain.

Weather is addressed at this stage for the following structural reasons:

1. Weather naturally exists at the daily level, which matches the defined analytical grain.
2. Weather is exogenous to traffic — it does not depend on traffic structure.
3. Aligning weather early allows us to validate temporal coverage across datasets.
4. Date alignment issues can be identified before merging more complex weekly or monthly datasets.
5. Weather integration introduces no facility-level conflicts, as it joins purely on DATE.

Cleaning the weather dataset before addressing weekly (Passenger, Bus) or monthly (Speeds) datasets ensures that the daily temporal framework is locked before more complex grain harmonization is performed.

This preserves integration clarity and prevents downstream restructuring.

In [7]:
# ==========================================
# TRAFFIC DATE RANGE CHECK
# ==========================================

min_date = traffic_daily["DATE"].min()
max_date = traffic_daily["DATE"].max()

print("Traffic Date Range:")
print("Start Date:", min_date)
print("End Date:", max_date)

Traffic Date Range:
Start Date: 2013-01-01 00:00:00
End Date: 2025-05-31 00:00:00


In [8]:
# ==========================================
# FACILITY LIST
# ==========================================

facilities = traffic_daily["FAC"].unique()

print("Facility Codes:")
print(sorted(facilities))

print("\nNumber of Facilities:", traffic_daily["FAC"].nunique())

Facility Codes:
[np.int64(1), np.int64(2), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]

Number of Facilities: 8


In [9]:
# If facility name columns exist in original traffic
traffic.groupby("FAC")[["FAC_G", "FAC_G2"]].first()

,FAC_G,FAC_G2
FAC,,
1,Tunnels,Holland
2,Tunnels,Lincoln
4,SIB,Bayonne
5,SIB,Goethals
6,SIB,Outerbridge
7,GWB,GWB
8,GWB,GWB
9,GWB,GWB


## Weather Dataset Requirements and Selection Logic

After finalizing the cleaned and aggregated traffic dataset (daily grain), the next step was to prepare the weather dataset for integration.

### 1. Why We Addressed Weather at This Stage

Before merging additional datasets, we needed to ensure that:

- The traffic dataset grain was stable (Daily level)
- The temporal range was confirmed
- The geographic coverage was consistent (NY/NJ metro region)

Since weather will be merged at the **DATE level**, we first validated:

- Minimum and maximum traffic dates
- That all traffic stations belong to the same metro region

This allowed us to define the required structure for the weather dataset:
> One row per DATE, covering the full traffic time range.

### 2. Dataset Selection

Instead of using an API pull, we downloaded an official daily weather dataset containing:

- STATION
- DATE
- AWND (Average Wind Speed)
- PRCP (Precipitation)
- SNOW (Snowfall)
- TMAX (Maximum Temperature)
- TMIN (Minimum Temperature)

This structure aligns with our daily traffic grain.

In [10]:
# =========================================
# WEATHER DATA – IMPORT & INSPECTION
# =========================================

import pandas as pd

# Load raw file
weather = pd.read_csv(
    "../data/raw/weather_raw.csv",
    parse_dates=["DATE"]
)

# Inspect structure
print("Shape:", weather.shape)
print("\nColumns:")
print(weather.columns)

print("\nHead:")
print(weather.head())

print("\nMissing Values:")
print(weather.isna().sum())

Shape: (4534, 7)

Columns:
Index(['STATION', 'DATE', 'AWND', 'PRCP', 'SNOW', 'TMAX', 'TMIN'], dtype='object')

Head:
       STATION       DATE  AWND  PRCP  SNOW  TMAX  TMIN
0  USW00094728 2013-01-01  31.0     0     0    44   -33
1  USW00094728 2013-01-02  26.0     0     0     6   -56
2  USW00094728 2013-01-03  20.0     0     0     0   -44
3  USW00094728 2013-01-04  36.0     0     0    28   -11
4  USW00094728 2013-01-05  30.0     0     0    56     0

Missing Values:
STATION      0
DATE         0
AWND       225
PRCP         0
SNOW         0
TMAX         0
TMIN         0
dtype: int64


In [11]:
# =========================================
# WEATHER CLEANING – FINAL STANDARDIZATION
# =========================================

# Convert DATE
weather["DATE"] = pd.to_datetime(weather["DATE"])

# Convert units
weather["TMAX"] = weather["TMAX"] / 10      # tenths °C → °C
weather["TMIN"] = weather["TMIN"] / 10
weather["PRCP"] = weather["PRCP"] / 10      # tenths mm → mm
weather["AWND"] = weather["AWND"] / 10      # tenths m/s → m/s

# Optional: Create average daily temperature
weather["TAVG"] = (weather["TMAX"] + weather["TMIN"]) / 2

# Check final structure
print(weather.head())
print("\nNew Data Types:")
print(weather.dtypes)

print("\nDate Range:")
print(weather["DATE"].min(), "to", weather["DATE"].max())

print("\nMissing Values:")
print(weather.isna().sum())

print("Missing %:")
print(weather.isna().mean() * 100)

weather.to_csv("../data/cleaned/weather_cleaned.csv", index=False)
print("Weather cleaned saved")

       STATION       DATE  AWND  PRCP  SNOW  TMAX  TMIN  TAVG
0  USW00094728 2013-01-01   3.1   0.0     0   4.4  -3.3  0.55
1  USW00094728 2013-01-02   2.6   0.0     0   0.6  -5.6 -2.50
2  USW00094728 2013-01-03   2.0   0.0     0   0.0  -4.4 -2.20
3  USW00094728 2013-01-04   3.6   0.0     0   2.8  -1.1  0.85
4  USW00094728 2013-01-05   3.0   0.0     0   5.6   0.0  2.80

New Data Types:
STATION            object
DATE       datetime64[ns]
AWND              float64
PRCP              float64
SNOW                int64
TMAX              float64
TMIN              float64
TAVG              float64
dtype: object

Date Range:
2013-01-01 00:00:00 to 2025-05-31 00:00:00

Missing Values:
STATION      0
DATE         0
AWND       225
PRCP         0
SNOW         0
TMAX         0
TMIN         0
TAVG         0
dtype: int64
Missing %:
STATION    0.000000
DATE       0.000000
AWND       4.962506
PRCP       0.000000
SNOW       0.000000
TMAX       0.000000
TMIN       0.000000
TAVG       0.000000
dtype: float

### Initial Inspection

We performed:

- Shape validation
- Column inspection
- Head preview
- Missing value analysis

Results:

- 4,534 rows
- 7 columns
- No missing values in DATE
- About 5% missing values in AWND
- No missing values in PRCP, SNOW, TMAX, TMIN


### 4. Structural Cleaning Decisions

The following actions were taken:

#### ✔ Converted DATE to datetime
Ensures merge compatibility with traffic dataset.

#### ✔ Verified Station Consistency
Confirmed all rows belong to a single station.
Since traffic is metro-level daily aggregation, station-level granularity is not required.

#### ✔ Retained Weather Variables
All weather variables were preserved because:
- They are meteorologically relevant
- They may contribute to downstream modeling


### 5. Missing Value Treatment Decision

AWND contains approximately 5% missing values.

Decision:
- Do not drop rows
- Do not impute at this stage
- Preserve NaN values

Reasoning:
- Missingness is low (<5%)
- AWND is not a merge key
- Imputation decisions belong to the modeling phase
- Structural dataset integrity is not affected


### 6. Columns Dropped

Since station-level differentiation is not required for daily metro-level merging:

- STATION column was dropped

Final structure before merge:

- DATE
- AWND
- PRCP
- SNOW
- TMAX
- TMIN


### 7. Final Prepared Weather Dataset

The cleaned weather dataset:

- Matches traffic dataset temporal grain (Daily)
- Contains no duplicate dates
- Contains no missing DATE values
- Is structurally ready for a left merge on DATE

The dataset was saved to:

`data/cleaned/weather_cleaned.csv`

---


In [12]:
print(weather.columns)

Index(['STATION', 'DATE', 'AWND', 'PRCP', 'SNOW', 'TMAX', 'TMIN', 'TAVG'], dtype='object')


In [13]:
print("Traffic date range:")
print(traffic_daily["DATE"].min(), "to", traffic_daily["DATE"].max())

Traffic date range:
2013-01-01 00:00:00 to 2025-05-31 00:00:00


In [14]:
# Merge weather into traffic backbone
traffic_weather = traffic_daily.merge(
    weather,
    on="DATE",
    how="left"
)

# Check structure
print("Shape after merge:", traffic_weather.shape)

# Check missing values after merge
print("\nMissing values after merge:")
print(traffic_weather.isna().sum())

traffic_weather.to_csv(
    "../data/cleaned/traffic_weather_merged.csv",
    index=False
)

Shape after merge: (35168, 17)

Missing values after merge:
FAC             0
DATE            0
TOTAL           0
CASH            0
EZPASS          0
VIOLATION       0
Autos           0
Small_T         0
Large_T         0
Buses           0
STATION         0
AWND         1792
PRCP            0
SNOW            0
TMAX            0
TMIN            0
TAVG            0
dtype: int64


In [15]:
print(traffic_weather.head())
print(traffic_weather.describe())

   FAC       DATE  TOTAL   CASH  EZPASS  VIOLATION  Autos  Small_T  Large_T  \
0    1 2013-01-01  35449   9824   24975        650  34978      197        3   
1    1 2013-01-02  41769   8620   32308        841  39778     1264       20   
2    1 2013-01-03  41744   7928   32995        821  39624     1390       25   
3    1 2013-01-04  42739   8673   33241        825  40618     1397       28   
4    1 2013-01-05  42187  10188   31212        787  41363      561        9   

   Buses      STATION  AWND  PRCP  SNOW  TMAX  TMIN  TAVG  
0    271  USW00094728   3.1   0.0     0   4.4  -3.3  0.55  
1    707  USW00094728   2.6   0.0     0   0.6  -5.6 -2.50  
2    705  USW00094728   2.0   0.0     0   0.0  -4.4 -2.20  
3    696  USW00094728   3.6   0.0     0   2.8  -1.1  0.85  
4    254  USW00094728   3.0   0.0     0   5.6   0.0  2.80  
                FAC                           DATE          TOTAL  \
count  35168.000000                          35168   35168.000000   
mean       5.169017  2019-0

In [16]:
import pandas as pd

# Load backbone (Traffic + Weather)
master = pd.read_csv(
    "../data/cleaned/traffic_weather_merged.csv",
    parse_dates=["DATE"]
)

# Load cleaned datasets (after audit)
passenger= pd.read_csv(
    "../data/processed/passenger_cleaned.csv"
)

bus= pd.read_csv(
    "../data/processed/bus_cleaned.csv"
)

speeds= pd.read_csv(
    "../data/processed/speeds_cleaned.csv"
)

print("Datasets Loaded")

Datasets Loaded


In [17]:
passenger["Start_Date"] = pd.to_datetime(passenger["Start_Date"])
passenger["End_Date"] = pd.to_datetime(passenger["End_Date"])

# Expand to daily
passenger_expanded = passenger.loc[
    passenger.index.repeat(
        (passenger["End_Date"] - passenger["Start_Date"]).dt.days + 1
    )
].copy()

passenger_expanded["DATE"] = passenger_expanded.groupby(level=0).cumcount()
passenger_expanded["DATE"] = (
    passenger_expanded["Start_Date"] +
    pd.to_timedelta(passenger_expanded["DATE"], unit="D")
)

# 🔥 CRITICAL FIX: Aggregate by DATE
passenger_daily = (
    passenger_expanded
    .groupby("DATE", as_index=False)["Volume"]
    .sum()
    .rename(columns={"Volume": "Passenger_Volume"})
)

print(passenger_daily.shape)

(1158, 2)


In [22]:

bus["Start_Date"] = pd.to_datetime(bus["Start_Date"])
bus["End_Date"] = pd.to_datetime(bus["End_Date"])

bus_expanded = bus.loc[
    bus.index.repeat(
        (bus["End_Date"] - bus["Start_Date"]).dt.days + 1
    )
].copy()

bus_expanded["DATE"] = bus_expanded.groupby(level=0).cumcount()
bus_expanded["DATE"] = (
    bus_expanded["Start_Date"] +
    pd.to_timedelta(bus_expanded["DATE"], unit="D")
)

# 🔥 CRITICAL FIX: Aggregate by DATE
bus_daily = (
    bus_expanded
    .groupby("DATE", as_index=False)["Volume"]
    .sum()
    .rename(columns={"Volume": "Bus_Volume"})
)

print(bus_daily.shape)

(1159, 2)


In [18]:


speeds["Month_Year"] = pd.to_datetime(
    speeds["Month_Year"],
    format="%Y-%m-%d"
)

speeds = speeds.rename(columns={"Facility": "FAC"})

# Keep needed columns
speeds = speeds[["FAC", "Month_Year", "Avg_Speed"]]

# Ensure unique rows
speeds = speeds.drop_duplicates()

print(speeds.shape)

(192, 3)


In [19]:
master = master.merge(
    passenger_daily,
    on="DATE",
    how="left"
)

print(master.shape)

(35168, 18)


In [25]:
master = master.merge(
    bus_daily,
    on="DATE",
    how="left"
)

print(master.shape)

(35168, 20)


In [26]:
print("master['FAC'] dtype :", master["FAC"].dtype)
print("speeds['FAC'] dtype :", speeds["FAC"].dtype)

master['FAC'] dtype : int64
speeds['FAC'] dtype : object


In [27]:
# Fix the FAC dtype on the DataFrame you're actually merging with
master["FAC"] = master["FAC"].astype(str)
speeds["FAC"] = speeds["FAC"].astype(str) 

In [28]:
keys = ["FAC", "Month_Year"]

print("Missing from master:", [k for k in keys if k not in master.columns])
print("Missing from speeds:", [k for k in keys if k not in speeds.columns])

print("\nmaster columns:", master.columns.tolist())
print("speeds columns:", speeds.columns.tolist())


Missing from master: ['Month_Year']
Missing from speeds: []

master columns: ['FAC', 'DATE', 'TOTAL', 'CASH', 'EZPASS', 'VIOLATION', 'Autos', 'Small_T', 'Large_T', 'Buses', 'STATION', 'AWND', 'PRCP', 'SNOW', 'TMAX', 'TMIN', 'TAVG', 'Passenger_Volume', 'Bus_Volume_x', 'Bus_Volume_y']
speeds columns: ['FAC', 'Month_Year', 'Avg_Speed']


In [29]:
import pandas as pd

# 0) Clean column names (avoid hidden spaces)
master.columns = master.columns.str.strip()
speeds.columns = speeds.columns.str.strip()

# 1) Ensure DATE is datetime
master["DATE"] = pd.to_datetime(master["DATE"])

# 2) Create Month_Year in master (monthly grain)
master["Month_Year"] = master["DATE"].values.astype("datetime64[M]")

# 3) Ensure speeds Month_Year is also monthly datetime
speeds["Month_Year"] = pd.to_datetime(speeds["Month_Year"], errors="coerce").values.astype("datetime64[M]")

# 4) Ensure FAC dtype matches (you already did this, but keep it consistent)
master["FAC"] = master["FAC"].astype(str)
speeds["FAC"] = speeds["FAC"].astype(str)

# 5) Merge
master = master.merge(speeds, on=["FAC", "Month_Year"], how="left")

print("After speeds merge:", master.shape)
print(master[["FAC", "DATE", "Month_Year", "Avg_Speed"]].head())


After speeds merge: (35168, 22)
  FAC       DATE Month_Year  Avg_Speed
0   1 2013-01-01 2013-01-01        NaN
1   1 2013-01-02 2013-01-01        NaN
2   1 2013-01-03 2013-01-01        NaN
3   1 2013-01-04 2013-01-01        NaN
4   1 2013-01-05 2013-01-01        NaN


In [30]:
print("Final Shape:", master.shape)
print("Duplicate rows:", master.duplicated().sum())

Final Shape: (35168, 22)
Duplicate rows: 0


In [31]:
master["AWND"] = master["AWND"].fillna(master["AWND"].median())

In [33]:
print("Bus_Volume in columns?", "Bus_Volume" in master.columns)
print(master.columns.tolist())


Bus_Volume in columns? False
['FAC', 'DATE', 'TOTAL', 'CASH', 'EZPASS', 'VIOLATION', 'Autos', 'Small_T', 'Large_T', 'Buses', 'STATION', 'AWND', 'PRCP', 'SNOW', 'TMAX', 'TMIN', 'TAVG', 'Passenger_Volume', 'Bus_Volume_x', 'Bus_Volume_y', 'Month_Year', 'Avg_Speed']


In [34]:
# See which one actually has values
print(master[["Bus_Volume_x", "Bus_Volume_y"]].isna().sum())
print(master[["Bus_Volume_x", "Bus_Volume_y"]].describe())

# Combine them (take y when present, otherwise x)
master["Bus_Volume"] = master["Bus_Volume_y"].combine_first(master["Bus_Volume_x"])

# Drop the suffixed columns
master = master.drop(columns=["Bus_Volume_x", "Bus_Volume_y"])

# Now your fill will work
master["Bus_Volume"] = master["Bus_Volume"].fillna(0)

print("Bus_Volume in columns now?", "Bus_Volume" in master.columns)


Bus_Volume_x    26641
Bus_Volume_y    26641
dtype: int64
       Bus_Volume_x  Bus_Volume_y
count   8527.000000   8527.000000
mean    2651.425237   2651.425237
std      470.624843    470.624843
min        0.000000      0.000000
25%     2578.500000   2578.500000
50%     2681.000000   2681.000000
75%     2910.000000   2910.000000
max     3097.000000   3097.000000
Bus_Volume in columns now? True


In [35]:
master["Passenger_Volume"] = master["Passenger_Volume"].fillna(0)
master["Bus_Volume"] = master["Bus_Volume"].fillna(0)

In [36]:
master["Avg_Speed"] = master["Avg_Speed"].fillna(master["Avg_Speed"].median())

In [37]:
print(master.isna().sum())
print("Duplicates:", master.duplicated().sum())

FAC                     0
DATE                    0
TOTAL                   0
CASH                    0
EZPASS                  0
VIOLATION               0
Autos                   0
Small_T                 0
Large_T                 0
Buses                   0
STATION                 0
AWND                    0
PRCP                    0
SNOW                    0
TMAX                    0
TMIN                    0
TAVG                    0
Passenger_Volume        0
Month_Year              0
Avg_Speed           35168
Bus_Volume              0
dtype: int64
Duplicates: 0


In [45]:
import pandas as pd

# 1) Standardize Month_Year on both
master["DATE"] = pd.to_datetime(master["DATE"])
master["Month_Year"] = master["DATE"].values.astype("datetime64[M]")

speeds["Month_Year"] = pd.to_datetime(speeds["Month_Year"], errors="coerce").values.astype("datetime64[M]")

# 2) Map master FAC numeric IDs -> speeds facility codes (VERIFY this mapping matches your facilities)
fac_map = {
    1: "HT",   # Holland Tunnel
    2: "LT",   # Lincoln Tunnel
    4: "BB",   # Bayonne Bridge
    5: "GB",   # Goethals Bridge
    6: "OBX",  # Outerbridge Crossing
    7: "GWB",  # George Washington Bridge
    8: "GWB",
    9: "GWB",
}

master["FAC_code"] = master["FAC"].map(fac_map).astype("string")
speeds["FAC_code"] = speeds["FAC"].astype("string").str.strip()

# 3) If speeds has multiple rows per month (e.g., EB/WB), collapse to one value per facility-month
speeds_monthly = (speeds
    .groupby(["FAC_code", "Month_Year"], as_index=False)["Avg_Speed"]
    .mean()
)

# 4) Merge in Avg_Speed cleanly
master = master.drop(columns=["Avg_Speed"], errors="ignore")
master = master.merge(speeds_monthly, on=["FAC_code", "Month_Year"], how="left")

# 5) Confirm it worked
print("Avg_Speed non-null:", master["Avg_Speed"].notna().sum(), "out of", len(master))
print("Avg_Speed % non-null:", round(master["Avg_Speed"].notna().mean() * 100, 2), "%")


Avg_Speed non-null: 0 out of 35168
Avg_Speed % non-null: 0.0 %


In [46]:
print(master.isna().sum())
print("Duplicates:", master.duplicated().sum())

FAC                     0
DATE                    0
TOTAL                   0
CASH                    0
EZPASS                  0
VIOLATION               0
Autos                   0
Small_T                 0
Large_T                 0
Buses                   0
STATION                 0
AWND                    0
PRCP                    0
SNOW                    0
TMAX                    0
TMIN                    0
TAVG                    0
Passenger_Volume        0
Month_Year              0
Bus_Volume              0
FAC_code            35168
Avg_Speed           35168
dtype: int64
Duplicates: 0


In [47]:
# Drop if it exists (won’t error if missing)
master = master.drop(columns=["Avg_Speed"], errors="ignore")

# Optional: also drop FAC_code if it’s fully missing
master = master.drop(columns=["FAC_code"], errors="ignore")


In [48]:
master.to_csv(
    "../data/integrated/final_master_dataset_cleaned.csv",
    index=False
)

print(" Final dataset saved.")

 Final dataset saved.


In [49]:
import pandas as pd

master = pd.read_csv(
    "../data/integrated/final_master_dataset_cleaned.csv",
    parse_dates=["DATE"]
)

print(master.shape)
master.head()

(35168, 20)


,FAC,DATE,TOTAL,CASH,EZPASS,VIOLATION,Autos,Small_T,Large_T,Buses,STATION,AWND,PRCP,SNOW,TMAX,TMIN,TAVG,Passenger_Volume,Month_Year,Bus_Volume
0,1,2013-01-01,35449,9824,24975,650,34978,197,3,271,USW00094728,3.1,0.0,0,4.4,-3.3,0.55,0.0,2013-01-01,0.0
1,1,2013-01-02,41769,8620,32308,841,39778,1264,20,707,USW00094728,2.6,0.0,0,0.6,-5.6,-2.50,0.0,2013-01-01,0.0
2,1,2013-01-03,41744,7928,32995,821,39624,1390,25,705,USW00094728,2.0,0.0,0,0.0,-4.4,-2.20,0.0,2013-01-01,0.0
3,1,2013-01-04,42739,8673,33241,825,40618,1397,28,696,USW00094728,3.6,0.0,0,2.8,-1.1,0.85,0.0,2013-01-01,0.0
4,1,2013-01-05,42187,10188,31212,787,41363,561,9,254,USW00094728,3.0,0.0,0,5.6,0.0,2.80,0.0,2013-01-01,0.0


In [50]:
import pandas as pd

obj_cols = master.select_dtypes(include=["object", "string"]).columns.tolist()
print("Object/String columns:", obj_cols)

audit = []
for c in obj_cols:
    s = master[c].astype("string")
    blank_n = (s.str.len() == 0).sum()                  # ""
    whitespace_only_n = (s.str.strip().str.len() == 0).sum()  # "   " (includes blanks)
    audit.append([c, int(blank_n), int(whitespace_only_n), int(master[c].isna().sum())])

audit_df = pd.DataFrame(audit, columns=["column", "blank_n", "whitespace_only_n", "na_n"])
display(audit_df.sort_values(["whitespace_only_n", "blank_n"], ascending=False))


Object/String columns: ['STATION', 'Month_Year']


,column,blank_n,whitespace_only_n,na_n
0,STATION,0,0,0
1,Month_Year,0,0,0


# Phase 2 Summary — Data Cleaning + Grain Alignment + Integration (Facility × Date)

## 1) What we started with (the problem)
We had multiple datasets that were **structurally sound** but recorded at **different granularities** (e.g., traffic at lane/time, passenger/bus weekly, speeds monthly, weather daily). If we merged them without a plan, we would risk **row inflation, misaligned joins, and distorted totals**. So we first chose one “final grain” to standardize everything around. 

---

## 2) The grain we chose (the “one-row-per…” rule)
We set the final analytical grain to:

**Facility × Date (Daily)**

This grain was chosen because:
- Traffic can be aggregated cleanly to daily facility totals.
- Weather aligns naturally to daily.
- Weekly passenger/bus can be expanded to daily.
- Monthly speeds can be mapped by month (if keys match correctly).
---

## 3) What we cleaned (in simple terms)
### **Traffic (Backbone)**
- Loaded the cleaned traffic dataset and confirmed its structure.
- Checked for duplicates at the operational grain (**FAC × LANE × DATE × TIME**).  
  We found duplicates but confirmed they were not simple copy/paste errors (values varied), so we **did not delete them**—we let aggregation handle consolidation.
- Validated internal consistency (e.g., `TOTAL` vs. payment components). Discrepancies were documented, but `TOTAL` was treated as authoritative because it aligned with vehicle category totals.

### **Weather**
- Loaded daily weather.
- Standardized units (tenths → standard units) and created `TAVG`.
- Kept missing wind (`AWND`) initially, then later filled missing values during final dataset cleaning. 

### **Passenger + Bus**
- Converted weekly ranges (Start_Date/End_Date) into daily rows.
- Aggregated to one row per `DATE`.
- Merged into the master using left joins, then filled missing Passenger/Bus volume with **0** (meaning: no recorded volume for that day). 

---

## 4) How we integrated everything (one master table)
We used the **Traffic Daily** table as the backbone and performed **left merges** so the master dataset always kept the same daily facility structure.
After each merge, we checked:
- Dataset shape (row count stability)
- Missing values (especially in newly added columns)
- Duplicate rows (to ensure no row inflation) 

---

## 5) Data quality checks we ran at the end
### **Missing values + duplicates**
We confirmed the final master dataset had **0 duplicate rows** after integration and cleaning. 

### **Zero-length / whitespace-only string check**
We audited all object/string columns to ensure there were **no empty strings** or **whitespace-only values** hiding as “fake” non-missing entries.

Result:
- Object/String columns detected: `STATION`, `Month_Year`
- `blank_n = 0`, `whitespace_only_n = 0`, `na_n = 0` for both columns (so string cleanliness checks passed).

---

## 6) What happened to the Avg_Speed (speed) column
We confirmed that the speeds source dataset contains a valid **Avg_Speed** field; however, when we tried to integrate Avg_Speed into the final master dataset, the merge produced **0 matched rows** — meaning:

**Avg_Speed ended up 100% missing (0 non-null values out of 35,168 rows).**

This indicates a **key alignment problem** between the master dataset and speeds dataset (e.g., facility identifiers/codes and/or monthly keys did not overlap correctly). Because there were **no real speed values** in the integrated table, **imputation was not possible** (you cannot compute a median from an entirely empty column). 

**Decision:** we treated Avg_Speed as *unusable in the integrated master* in its current form, documented the issue, and proceeded without it (or kept the speeds dataset separate for future correction).

---

## 7) Final result (what we saved)
At the end of Phase 2, we produced and saved a **clean, integrated, daily Facility × Date dataset** that is ready for the next phase (sampling + modeling). The dataset has coherent grain alignment, validated joins, strong missing-value handling, and clean string fields.